# Resumable factuality geometry on Kaggle

Start with the pilot. Fit methods without truthfulness labels, then evaluate saved predictions. Enable Internet for installation/model downloads and a GPU for encoding. Attach the author dataset; this notebook does not download or republish it.

Snapshots are local until you persist/download notebook outputs. A discarded Kaggle disk cannot be recovered automatically. Save `study-checkpoint.tar.gz` between bounded runs.


In [ ]:
from pathlib import Path
import json, subprocess, sys
REPO = Path('/kaggle/working/factuality-geometry')
OUTPUT = Path('/kaggle/working/study-runs')
SNAPSHOT = Path('/kaggle/working/study-checkpoint.tar.gz')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','v0.1.0',
                    'https://github.com/Ram21275/factuality-geometry.git', str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(REPO)+'[encoders]'], check=True)
print('Git commit:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())


## Inspect inputs and choose your table

Set `DATA` after reading this file inventory. For separate release tables, normalize claims and explicitly join evidence/image mappings. Ruling/justification fields must not become model inputs.


In [ ]:
for path in sorted(Path('/kaggle/input').rglob('*')):
    if path.is_file() and path.suffix in {'.csv','.tsv','.parquet','.json','.jsonl','.zip','.gz'}:
        print(path, path.stat().st_size)
DATA = Path('/kaggle/input/YOUR_DATASET/YOUR_CLAIMS.csv')  # EDIT
if DATA.exists():
    from fgeometry.data import read_table
    print('Columns:', list(read_table(DATA).columns))


## Optional restoration

In a new session attach your saved snapshot as an input. Set `RESTORE_ARCHIVE` below. Restoration requires an empty output directory and refuses to overwrite existing work. Reuse the same code and scientific package versions; their hashes are recorded in the manifest.


In [ ]:
RESTORE_ARCHIVE = None  # e.g. '/kaggle/input/my-study-output/study-checkpoint.tar.gz'
if RESTORE_ARCHIVE and (not OUTPUT.exists() or not any(OUTPUT.iterdir())):
    subprocess.run([sys.executable,'-m','fgeometry.cli','restore',
                    '--archive',RESTORE_ARCHIVE,'--output',str(OUTPUT)], check=True)


## Configure the pilot

Column names and raw verdict spellings need checking against your release. Numeric labels require an explicit map. NEI stays unmapped for the binary evaluation, but remains in label-free fitting. Set `split_mode` to `group`, `official`, `topic`, or `source`. Do not tune parameters using test results.


In [ ]:
CONFIG = Path('/kaggle/working/study-config.json')
cfg = json.loads((REPO/'configs/pilot.json').read_text())
# EDIT these mappings after inspecting the dataset. Automatic aliases work for canonical columns.
cfg['columns'] = {}  # e.g. {'id':'claim_id','text':'claim_text','label':'truthfulness'}
cfg['label_map'] = {'supported':1,'support':1,'refuted':0,'refute':0,'nei':None,
                    'not enough information':None}
cfg['split_mode'] = 'group'
# For a restored run, recover the exact original configuration.
if RESTORE_ARCHIVE:
    manifests = sorted(OUTPUT.glob('*/manifest.json'))
    if len(manifests) == 1:
        cfg = json.loads(manifests[0].read_text())['config']
    else:
        print('Multiple saved runs: choose the corresponding manifest/config explicitly.')
CONFIG.write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))


## Run one bounded chunk; repeat this cell to resume

The soft budget stops after the current unit has committed. Completed shards/jobs are reused. The full study is intentionally split across sessions. Avoid starting two processes against the same output directory.


In [ ]:
assert DATA.exists(), 'Set DATA to your actual attached claim table first'
STAGES = 'audit,embed,experiments,geometry,evaluate'
MAX_SECONDS = 3300
command = [sys.executable,'-m','fgeometry.cli','run','--data',str(DATA),
           '--config',str(CONFIG),'--output',str(OUTPUT),'--stages',STAGES,
           '--max-seconds',str(MAX_SECONDS),'--snapshot-to',str(SNAPSHOT)]
subprocess.run(command, check=True)
print('Persist or download this archive before discarding the session:', SNAPSHOT)


## Inspect complete reports

A paused run may not have reached evaluation yet. Repeating the run continues remaining units. AUC* is orientation-free label association, not a deployed fake/true score. Inspect every prespecified result rather than selecting the best test-scoring configuration.


In [ ]:
import pandas as pd
for report in sorted(OUTPUT.glob('*/report.json')):
    result = json.loads(report.read_text())
    print(report, result['semantic_orientation'])
    display(pd.DataFrame(result['results']).drop(columns=['per_topic'], errors='ignore'))
for report in sorted(OUTPUT.glob('*/paired_comparisons.json')):
    display(pd.DataFrame(json.loads(report.read_text())))
for report in sorted(OUTPUT.glob('*/geometry_report.json')):
    display(pd.DataFrame(json.loads(report.read_text())))


## Optional branches after the text audit

Evidence requires explicit `evidence_texts` lists; images require `image_paths` and a local image root. Run every modality on the same paired records. Screenshots/rating badges require leakage auditing. Example: set `STAGES` to `evidence,evaluate`, or `multimodal,evaluate` and add `--image-root` to the command.

For confirmed multi-session experiments replace the pilot with `configs/full.json`, keeping the encoder revision and shard size fixed. Shared cached embeddings can be reused. Treat topic/source holdouts and encoder replication as separate named configurations.


## Export a fresh checkpoint whenever needed

Persist this archive as a saved notebook output or download it. Reattach the saved archive next session and use the restoration cell. No automatic external uploads or credentials are used.


In [ ]:
subprocess.run([sys.executable,'-m','fgeometry.cli','snapshot',
                '--output',str(OUTPUT),'--destination',str(SNAPSHOT)], check=True)
print(SNAPSHOT, SNAPSHOT.stat().st_size, 'bytes')
